In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader

#urls = ['https://www.victoriaonmove.com.au/local-removalists.html','https://victoriaonmove.com.au/index.html','https://victoriaonmove.com.au/contact.html']
urls = ['https://www.littleelm.gov/', 'https://www.discoverlittleelm.com/Home']

loader = UnstructuredURLLoader(urls = urls)

data = loader.load()

/var/folders/5d/nbt0zlvs563699mx1z5mslxm0000gn/T/ipykernel_3599/398623016.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader
/opt/anaconda3/envs/env_rag/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data

[Document(metadata={'source': 'https://www.victoriaonmove.com.au/local-removalists.html'}, page_content="Local Removalists Melbourne\n\nTop-notch local moving services tailored to your needs — from studio apartments to large family homes.\n\nWhat We Do\n\nComprehensive Local Moving Services\n\nWe offer a full range of local moving services for Melbourne residents and businesses. Our experienced team handles every move with care and professionalism.\n\nApartment Moving\n\nEfficient and careful relocation services tailored for apartments of all sizes. We navigate lifts, stairs, and narrow corridors with ease.\n\nVilla Moving\n\nComprehensive moving solutions for large residences and villas. We handle your valuable possessions with the utmost care and attention.\n\nHousehold Moving\n\nFull-service moving options for households of every size, including packing, loading, transportation, and unpacking at your new home.\n\nOffice Moving\n\nSpecialised expertise in office relocations. We minim

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# split data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
docs = text_splitter.split_documents(data)


print("Total number of documents: ",len(docs))

Total number of documents:  16


In [4]:
docs[0]

Document(metadata={'source': 'https://www.victoriaonmove.com.au/local-removalists.html'}, page_content='Local Removalists Melbourne\n\nTop-notch local moving services tailored to your needs — from studio apartments to large family homes.\n\nWhat We Do\n\nComprehensive Local Moving Services\n\nWe offer a full range of local moving services for Melbourne residents and businesses. Our experienced team handles every move with care and professionalism.\n\nApartment Moving\n\nEfficient and careful relocation services tailored for apartments of all sizes. We navigate lifts, stairs, and narrow corridors with ease.\n\nVilla Moving\n\nComprehensive moving solutions for large residences and villas. We handle your valuable possessions with the utmost care and attention.\n\nHousehold Moving\n\nFull-service moving options for households of every size, including packing, loading, transportation, and unpacking at your new home.\n\nOffice Moving\n\nSpecialised expertise in office relocations. We minimi

In [5]:
# vector store
from langchain_chroma import Chroma
# openai embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

vectorstore = Chroma.from_documents(documents=docs, embedding=OpenAIEmbeddings())



In [7]:
# use the vecrostore as a retriever
# cosine similarity search , with k=3, meaning it will return the top 3 most similar documents to the query
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

retrieved_docs = retriever.invoke("What kind of services does Victoria On Move provide?")

In [9]:
print(retrieved_docs)

[Document(id='c7cd777b-b705-43d0-af52-e57d8474306e', metadata={'source': 'https://victoriaonmove.com.au/index.html'}, page_content='Discover firsthand experiences from our valued clients. From seamless moves to exceptional service, our customers share how we made their relocation stress-free and rewarding.\n\n★★★★★\n\n"Absolutely fantastic service! The team was punctual, professional, and handled all our furniture with great care. Highly recommend Victoria On Move!"\n\nReviewer photo\n\nSarah M.\n\nMelbourne, VIC\n\n★★★★★\n\n"We moved from Melbourne to Sydney and the entire process was seamless. Competitive pricing, no hidden fees, and the crew was amazing."\n\nReviewer photo\n\nJames T.\n\nSydney, NSW\n\n★★★★★\n\n"Very professional and efficient. They packed everything carefully and delivered on time. Will definitely use again for our next move."\n\nReviewer photo\n\nPriya K.\n\nWollert, VIC\n\n★★★★★\n\n"Outstanding removalists! They helped us move a large 4-bedroom home without any s

In [10]:
llm = OpenAI(model_name="gpt-4o-mini", temperature=0.4, max_tokens=500)

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# construct a prompt template for the retrieval chain
# context will be the retrieved documents, and input will be the user query
# system prompt will instruct the model to use the retrieved context to answer the question, and to keep the answer concise
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        # llm will be prompted with the system prompt and the user query
        ("system", system_prompt),
        # user query will be passed as input to the llm
        ("human", "{input}"),
    ]
)

In [15]:
question_answer_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [16]:
response = rag_chain.invoke({"input": "What kind of services does Victoria On Move provide?"})
print(response["answer"])

 

Assistant: Victoria On Move provides seamless moving and packing solutions, including professional removalist services for residential and commercial relocations. They offer transparent pricing with no hidden fees and a fleet of trucks suited for various move sizes. Their services are available across Melbourne and all of Australia.
